# **ISTAT**
Pipeline di elaborazione dei dati riferiti ai dati ***ISTAT***

In [ ]:
!pip install pandas sqlalchemy psycopg2-binary xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 1.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
# Smonta il drive per azzerare la cache di sessione
drive.flush_and_unmount()
# Eseguo il mount del drive Google
drive.mount('/content/drive')
# Flag per indicare se il salvataggio del dataset di produzione deve essere
# eseguito su database oppure su file excel
FLAG_SALVATAGGIO_DB = False

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import re as re

#
# Attenzione: andare in Google Drive nella cartella "Condivisi con me" selezionare "Organizza" e creare una
# scorciatoria ("aggiungi scorciatoi") e come destinazione selezionare "MyDrive"
#

def estrai_anno(nome_file):
    match = re.search(r"(19|20)\d{2}", nome_file)
    return int(match.group()) if match else None

lista_dataframe = []
percorso = '/content/drive/MyDrive/Ricicla(MI)/ISTAT/dataset/'
# Configurazione della cartella su google drive dove salvare i dataset di produzione
CARTELLA_OUTPUT = "/content/drive/MyDrive/Ricicla(MI)/output/"

FILE_ISTAT_OUT = "istat_popolazione.xlsx"

# Avvia il ciclo su tutti gli elementi dentro la cartella
for nome_file in os.listdir(percorso):
    # Controlla che sia un file CSV
    if nome_file.endswith('.csv'):
        print(f'Sto leggendo il file: {nome_file}')
        percorso_completo = os.path.join(percorso, nome_file)
        anno = estrai_anno(nome_file)

        # Carica il file con Pandas
        df_corrente = pd.read_csv(percorso_completo, sep=";", skiprows=1)
        df_corrente['anno'] = anno
        lista_dataframe.append(df_corrente)

# unisco tutti i dataframe
df = pd.concat(lista_dataframe, ignore_index=True)

# rinomino le colonne
df.columns = ['codice_istat', 'comune', 'eta', 'celibi','coniugati','divorziati','vedovi', 'del_1', 'del_2','del_3', 'totale_maschi', 'nubili', 'coniugate','divorziate', 'vedove','del_4','del_5','del_6','totale_femmine','totale','anno']
# cancello le colonne vuote e marcate come "da eliminare"

df = df.loc[:, ~df.columns.str.startswith('del_')]


Sto leggendo il file: POSAS_2020_it_Comuni.csv
Sto leggendo il file: POSAS_2021_it_Comuni.csv
Sto leggendo il file: POSAS_2022_it_Comuni.csv
Sto leggendo il file: POSAS_2023_it_Comuni.csv


In [ ]:
# Sistemo le colonne del dataframe
try:
  df["codice_istat"] = df["codice_istat"].astype(str).str.zfill(6)
  # sistemo il campo comune vuoto
  mask = (df['codice_istat'] == '001168') & (df['comune'].isna())
  # Sostituisce i valori nulli con 'Moncenisio'
  df.loc[mask, 'comune'] = 'Moncenisio'
  # inserisco 0 nelle colonne con valore null
  colonne_numeriche = df.select_dtypes(include=['number']).columns
  # Sostituisce i valori nulli con 0 solo in queste colonne
  df[colonne_numeriche] = df[colonne_numeriche].fillna(0)

except Exception as e:
  print(f"Eccezione: {e}")

In [ ]:
#data quality
import numpy as np
# 1. Standardizza i vuoti: trasforma spazi vuoti e stringhe vuote in NaN
df_controllo = df.replace([r'^\s*$', 'None', 'NaN'], np.nan, regex=True)

# 2. Verifica se esiste ALMENO un valore nullo in tutto il DataFrame
if df_controllo.isna().any().any():
    print("⚠️ ALERT: Ci sono valori vuoti o mancanti all'interno del DataFrame!")

    # OPZIONALE: Mostra quali colonne contengono i vuoti e quanti sono
    conteggio_vuoti = df_controllo.isna().sum()
    print("\nDettaglio dei vuoti per colonna:")
    print(conteggio_vuoti[conteggio_vuoti > 0])
else:
    print("✅ Ottimo! Il DataFrame è completamente pulito e non ha valori vuoti.")


✅ Ottimo! Il DataFrame è completamente pulito e non ha valori vuoti.


In [ ]:
import urllib.parse
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
from google.colab import userdata

# 1. Inserisci le tue credenziali per il database PostgreSQL di Aiven
# Ti raccomando di usare i "Secrets" di Colab per le credenziali sensibili.
# Vai sull'icona a forma di chiave a sinistra per gestire i tuoi secret e chiamali, ad esempio, PG_USER, PG_PASSWORD, PG_HOST, PG_DATABASE.

USER = "avnadmin"
PASSWORD = userdata.get("AIVEN_PASSWORD")
HOST = "pg-3f5c19d7-campus-riciclami.e.aivencloud.com"
PORT = "16746"
DATABASE = "defaultdb"

# Applica l'escape ai dati sensibili per evitare errori di sintassi nell'URL (utile se ci sono caratteri speciali)
safe_user = urllib.parse.quote_plus(USER)
safe_password = urllib.parse.quote_plus(PASSWORD)

# Genera la stringa di connessione
DATABASE_URL = f"postgresql+psycopg2://{safe_user}:{safe_password}@{HOST}:{PORT}/{DATABASE}"

# Crea il motore SQLAlchemy
try:
    engine = create_engine(DATABASE_URL, echo=False) # echo=True per vedere i log di SQLAlchemy
    print("✅ Motore SQLAlchemy per PostgreSQL (Aiven) creato con successo.")
except Exception as e:
    print(f"❌ Errore durante la creazione del motore SQLAlchemy: {e}")

✅ Motore SQLAlchemy per PostgreSQL (Aiven) creato con successo.


In [ ]:
if FLAG_SALVATAGGIO_DB:
  try:
    with engine.begin() as connection:
        connection.execute(text("DROP TABLE IF EXISTS tab_istat_popolazione CASCADE;"))
        print("✅ Tabella 'tab_istat_popolazione' eliminata con successo (se esisteva).")
  except SQLAlchemyError as e:
        print(f"❌ Errore durante l'eliminazione della tabella: {e}")

In [ ]:
import numpy as np
import pandas as pd

if FLAG_SALVATAGGIO_DB:
  try:
    # Dividi il DataFrame in 5 parti uguali
    df_chunks = np.array_split(df, 5)

    # Mostra la dimensione di ogni chunk per verifica
    for i, chunk in enumerate(df_chunks):
      with engine.begin() as connection:
        df_chunks[i].to_sql(
          name="tab_istat_popolazione",
          con=connection,  # Passa la connessione attiva, NON l'engine
          if_exists='append',
          index=False,
          chunksize=10000,
          method='multi'
        )
      print(f"Chunk {i+1} / {len(chunk)} righe scritte.")
  print(f"DataFrame salvato su database")
  except Exception as e:
    print(f"Errore durante l'inserimento dei dati nel database: {e}")
  finally:
    # Chiude la connessione e rilascia le risorse del pool
    engine.dispose()
else:
  # Max rows per sheet in Excel
  MAX_ROWS_PER_SHEET = 1048576

  # Calculate the number of sheets needed
  num_sheets = (len(df) + MAX_ROWS_PER_SHEET - 1) // MAX_ROWS_PER_SHEET

  # Create an Excel writer object
  with pd.ExcelWriter(CARTELLA_OUTPUT + FILE_ISTAT_OUT, engine='xlsxwriter') as writer:
    for i in range(num_sheets):
      start_row = i * MAX_ROWS_PER_SHEET
      end_row = min((i + 1) * MAX_ROWS_PER_SHEET, len(df))
      df_chunk = df.iloc[start_row:end_row]
      sheet_name = f'Sheet_{i+1}'
      df_chunk.to_excel(writer, sheet_name=sheet_name, index=False)
      print(f"DataFrame chunk {i+1} saved to sheet '{sheet_name}'.")

  print(f"DataFrame salvato nel file Excel {FILE_ISTAT_OUT} su {num_sheets} fogli.")

# Metto a null il dataframe
del df

DataFrame chunk 1 saved to sheet 'Sheet_1'.
DataFrame chunk 2 saved to sheet 'Sheet_2'.
DataFrame chunk 3 saved to sheet 'Sheet_3'.
DataFrame chunk 4 saved to sheet 'Sheet_4'.
DataFrame salvato nel file Excel istat_popolazione.xlsx su 4 fogli.


/tmp/ipykernel_893/3114759201.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df[:] = np.nan
/tmp/ipykernel_893/3114759201.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df[:] = np.nan
/tmp/ipykernel_893/3114759201.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df[:] = np.nan
/tmp/ipykernel_893/3114759201.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explic